# 21 — Monte Carlo doses of the frozen virtual cohort

**What this notebook does.** The full Monte Carlo run of nb15's design finished on 2026-11-10 (10 PET-weighted sources per phantom, 5×10⁵ decays each, 227 phantoms). Here:
1. integrity of the run: source count, self-dose against mass-scaled S-values, reciprocity of cross-doses;
2. link of the run to the frozen cohort: TIA per Monte Carlo source, voxel basis per phantom, per-lesion sources through the native split kernel;
3. organ, lesion and voxel doses for every virtual patient and scenario, both splits;
4. comparison with the self-dose S-value doses that nb19 calibrated on, per organ and per lesion.

**Test split.** Doses of the test cohort are computed and saved, as the network will need them at evaluation, but **no statistic of them is shown or used**: only counts. Comparisons are on the training split.

**The network is not started here.**

**About this notebook.** Monte Carlo doses of the frozen cohort: integrity, self-dose, reciprocity, gland link, metal, voxel maps.
- **Inputs:** `mc_full/` (`scripts/monte_carlo/run_mc_full.py`, 5 days CPU), basis by `scripts/monte_carlo/run_cohort_doses.py` (about 8 h), cohort.
- **Outputs:** `data/processed/psma/cohort_doses/`.
- **Run time:** about 7 min with the basis built.
- **Environment:** .venv.

In [ ]:
import os
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
import hashlib
import itertools
import json
import subprocess
import sys
from multiprocessing import Pool
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(ROOT / "src"))
import cohort_doses as D
import pbpk_calibration as C

M = D.M
OUT = D.OUT
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=ROOT).stdout.strip()
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
INK, MUTED = "#0b0b0b", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 9, "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "axes.edgecolor": "#b8b7ae"})
pd.set_option("display.width", 230, "display.max_columns", 40, "display.max_rows", 150)
print("commit", COMMIT)

manifest = pd.read_csv(ROOT / "data/processed/psma/cohort_manifest.csv")
bad = [r.file for r in manifest.itertuples() if hashlib.sha256((D.COHORT / r.file).read_bytes()).hexdigest() != r.sha256]
print("frozen cohort tables matching the manifest:", len(manifest) - len(bad), "of", len(manifest), bad)

split_table = pd.read_csv(M.ph.psma_data.PROCESSED / "psma_68ga_split.csv")
STUDIES = sorted(split_table[split_table.split.isin(["train", "test"])].study_id)
SPLIT_OF = split_table.set_index("study_id").split
if not (OUT / "phantom_targets.csv").exists():
    with Pool(5) as pool:
        result = pool.map(D.phantom_info, STUDIES, chunksize=4)
    pd.DataFrame([r for a, _, _ in result for r in a]).to_csv(OUT / "phantom_targets.csv", index=False)
    pd.DataFrame([e for _, e, _ in result]).to_csv(OUT / "phantom_info.csv", index=False)
    pd.DataFrame([r for _, _, l in result for r in l]).to_csv(OUT / "phantom_lesions.csv", index=False)
targets_info = pd.read_csv(OUT / "phantom_targets.csv")
info = pd.read_csv(OUT / "phantom_info.csv").set_index("study_id")
lesion_info = pd.read_csv(OUT / "phantom_lesions.csv")
mass = targets_info.set_index(["study_id", "target"]).mass_g
print(len(STUDIES), "phantoms:", (SPLIT_OF[STUDIES] == "train").sum(), "train,", (SPLIT_OF[STUDIES] == "test").sum(), "test")

## 1. Integrity of the run

**Source count.** 227 phantoms × 10 sources = 2270 runs. A source is skipped, with a `skip_<source>` file, only when its PET weights are empty. Expected: the 24 lesion-free studies have no lesion source, and one study has no bladder source. `run.lock` was left by the finished launch (pid 1987, no process alive, lock released by the kernel); it was removed before this notebook.

In [ ]:
status = json.loads((D.MC_FULL / "status.json").read_text())
print({k: status[k] for k in ("current", "source_runs_done", "source_runs_total", "n_per_source", "failed_this_launch")})
skips = sorted((p.parent.name, p.name[5:]) for p in D.MC_FULL.glob("psma_*/skip_*"))
done = sorted((p.parent.name, p.name[2:-11]) for p in D.MC_FULL.glob("psma_*/D_*_pet.nii.gz"))
print("source maps on disk", len(done), "| skipped", len(skips), "| sum", len(done) + len(skips), "| expected", 10 * len(STUDIES))
print("lock file present:", (D.MC_FULL / "run.lock").exists())
lesion_free = sorted(s for s in STUDIES if json.loads((M.ph.PHANTOMS / s / "meta.json").read_text())["lesions"]["n"] == 0)
skip_lesions = sorted(s for s, c in skips if c == "lesions")
print("skip_lesions", len(skip_lesions), "| lesion-free phantoms", len(lesion_free), "| same set:", skip_lesions == lesion_free)
other = [(s, c) for s, c in skips if c != "lesions"]
print("other skips:", other)
for s, c in other:
    organs, pet = M.load(s, "organs"), M.load(s, "pet_pct_ia_ml")
    mask = organs == 9
    meta = json.loads((M.ph.PHANTOMS / s / "meta.json").read_text())
    print(f"  {s}: bladder voxels {mask.sum()}, PET in them {float(pet[mask].sum()):.3g}, PET crop k_min of the original {meta['grid']['k_min_of_original']}")
runs = pd.read_csv(D.MC_FULL / "targets.csv")
print("targets.csv rows by kind:", runs.kind.value_counts().to_dict(), "| duplicated (study, run):", int(runs.duplicated(["study_id", "run"]).sum()))
print("events per source run: min", runs[runs.kind == "source"].n.min(), "max", runs[runs.kind == "source"].n.max())
failures = pd.read_csv(D.MC_FULL / "failures.csv")
retried = [(r.study_id, r.run) for r in failures.itertuples() if (D.MC_FULL / r.study_id / f"D_{r.run}_pet.nii.gz").exists()]
print("failures logged", len(failures), "of which redone later", len(retried))

## 2. Self-dose against mass-scaled S-values

For a source $c$ whose target is the same set of voxels, $D(c\leftarrow c)\,m_c$ (mGy·g per MBq·h) is compared with $S_{ref}\,m_{ref}$: EANM 2022 Tables 2 and 5 (IDAC-Dose 2.1 adult male, as nb14) for kidneys, liver, spleen and glands; for the other sources the electron energy per decay $\Delta_e$ = 85.3 mGy·g/(MBq·h) (ICRP 107), i.e. local deposition. Masses are those of the Monte Carlo geometry (ICRP 110 densities in the overridden organs, Schneider elsewhere).

What to expect: about 1.00-1.02 for organs (electrons absorbed, a few percent of photons on top); below that for very small structures (electrons escape); for large ones up to $1+\Delta_\gamma/\Delta_e$ = 1.23 if all photons were absorbed. Two sources do not have their own target. **Spine:** the source is the whole spine (labels 12 and 13), the target only L2-L4 (label 12), so the ratio is divided by the share of the spine PET that lies in L2-L4. **Rest of body:** the source holds labels 2 and 14 with the shells set to the median, the target label 14 only. Lesions: grouped source on all lesion voxels.

Outliers: robust z (median, 1.4826 MAD of log ratio) above 4 in absolute value, per source; plus the four cases flagged before this notebook.

In [ ]:
source = D.source_table()
SELF_TARGET = {c: c for c in D.SOURCES}
SELF_TARGET["spine"] = "spine_L2_L4"
self_dose = source[[SELF_TARGET[c] == t for c, t in zip(source.source, source.target)]].copy()
self_dose["mass_g"] = [mass[(s, t)] for s, t in zip(self_dose.study_id, self_dose.target)]
self_dose["volume_ml"] = [targets_info.set_index(["study_id", "target"]).volume_ml[(s, t)] for s, t in zip(self_dose.study_id, self_dose.target)]
self_dose["d_times_m"] = self_dose.d * self_dose.mass_g * 1e3
self_dose["reference"] = [D.EANM_SELF[c][0] * D.EANM_SELF[c][1] if c in D.EANM_SELF else D.DELTA_E_MJ * 1e3 for c in self_dose.source]
self_dose["reference_kind"] = ["EANM S-value" if c in D.EANM_SELF else "local deposition" for c in self_dose.source]
self_dose["share"] = [info.spine_pet_in_l2l4[s] if c == "spine" else 1.0 for s, c in zip(self_dose.study_id, self_dose.source)]
self_dose["ratio"] = self_dose.d_times_m / (self_dose.reference * self_dose.share)
self_dose["ratio_naive"] = self_dose.d_times_m / self_dose.reference


def robust_z(x):
    log = np.log(x)
    return (log - np.median(log)) / (1.4826 * np.median(np.abs(log - np.median(log))))


self_dose["z"] = self_dose.groupby("source").ratio.transform(robust_z)
summary = self_dose.groupby("source").agg(n=("ratio", "size"), reference=("reference_kind", "first"), p01=("ratio", lambda v: v.quantile(0.01)), median=("ratio", "median"), p99=("ratio", lambda v: v.quantile(0.99)), outliers_z4=("z", lambda v: int((v.abs() > 4).sum())), r_with_log_mass=("ratio", lambda v: np.corrcoef(v, np.log(self_dose.loc[v.index, "mass_g"]))[0, 1]))
display(summary.round(3))
self_dose.to_csv(OUT / "self_dose_vs_s_values.csv", index=False)
FLAGGED = [("psma_8effd73b76ee3191_2016-09-24", "bladder"), ("psma_4aa52e14d0947256_2017-09-16", "salivary_glands"), ("psma_bc63defabeea583e_2017-04-22", "spine"), ("psma_e51e9429c79eff86_2015-01-09", "spine")]
flagged = self_dose.set_index(["study_id", "source"]).loc[FLAGGED, ["mass_g", "volume_ml", "share", "ratio_naive", "ratio", "z"]]
flagged["source_median"] = [self_dose[self_dose.source == c].ratio.median() for _, c in FLAGGED]
flagged["mass_rank"] = [int((self_dose[self_dose.source == c].mass_g < m).sum()) + 1 for (_, c), m in zip(FLAGGED, flagged.mass_g)]
print("The four flagged cases (mass_rank 1 = smallest of its source in the cohort)")
display(flagged.round(4))
print("All |z| > 4 with the definitions above")
display(self_dose[self_dose.z.abs() > 4][["study_id", "source", "mass_g", "ratio", "z"]].round(3))

In [ ]:
spine_naive = self_dose[self_dose.source == "spine"]
print("spine, naive ratio: quantiles 1/50/99%", spine_naive.ratio_naive.quantile([0.01, 0.5, 0.99]).round(3).to_list(), "| rank of the two flagged among 227 (1 = lowest):", [int((spine_naive.ratio_naive < spine_naive.set_index("study_id").ratio_naive[s]).sum()) + 1 for s in ("psma_bc63defabeea583e_2017-04-22", "psma_e51e9429c79eff86_2015-01-09")])
for s in ("psma_bc63defabeea583e_2017-04-22", "psma_e51e9429c79eff86_2015-01-09"):
    meta = json.loads((M.ph.PHANTOMS / s / "meta.json").read_text())
    c = meta["compartments"]
    print(f"  {s}: SUV L2-L4 {c['spine_L2_L4']['mean_suv']:.2f}, other spine {c['spine_other']['mean_suv']:.2f}, PET share in L2-L4 {info.spine_pet_in_l2l4[s]:.3f}, lesions {meta['lesions']['n']}")
print("cohort PET share in L2-L4: quantiles 1/50/99%", info.spine_pet_in_l2l4.quantile([0.01, 0.5, 0.99]).round(3).to_list())
meta = json.loads((M.ph.PHANTOMS / "psma_4aa52e14d0947256_2017-09-16" / "meta.json").read_text())
print("glands of psma_4aa52e14d0947256_2017-09-16:", {k: (round(v["volume_ml"], 2), v["truncated"], v["use_for_pbpk"]) for k, v in meta["salivary_glands"].items()})
meta = json.loads((M.ph.PHANTOMS / "psma_8effd73b76ee3191_2016-09-24" / "meta.json").read_text())
print("bladder of psma_8effd73b76ee3191_2016-09-24:", round(meta["compartments"]["urinary_bladder"]["volume_ml"], 2), "mL; nb09 flags", meta["qc_flags"]["nb09_flags"])

fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharey=False)
for ax, c in zip(axes, ["bladder", "salivary_glands", "kidneys_functional", "spine"]):
    part = self_dose[self_dose.source == c]
    ax.scatter(part.mass_g, part.ratio, s=9, color=COLORS[0], alpha=0.6, linewidths=0)
    for s, cc in FLAGGED:
        if cc == c:
            row = part[part.study_id == s]
            ax.scatter(row.mass_g, row.ratio, s=40, facecolor="none", edgecolor=COLORS[1], linewidths=1.5)
    ax.set_xscale("log")
    ax.set_title(c + (" (÷ PET share in L2-L4)" if c == "spine" else ""), fontsize=9)
    ax.set_xlabel("target mass (g)")
axes[0].set_ylabel("D(c←c)·m / mass-scaled reference")
fig.tight_layout()
fig.savefig(OUT / "self_dose_vs_mass.png", dpi=150)
plt.show()

**Reading.** The run is complete and self-consistent: 2245 source maps plus 25 expected skips (the 24 lesion-free studies and the bladder of the cropped study), 499 996-499 998 events per run, the 5 logged failures all redone.

- **Organs with an S-value** (kidneys, liver, spleen, glands): median 1.006-1.012, 1st-99th percentile 0.995-1.022. Above 1 by about 1% as expected (photons on top of the absorbed electrons); the ratio rises with mass (r 0.73-0.92 with log mass), as photon absorption does.
- **Local-deposition references:** blood pool 1.006, spine 1.022 (after the L2-L4 share), bladder 1.024, GI 1.027, rest of body 1.094 (up to 1.22). The rest of body is a 35-65 kg source, so a large part of the photon energy is absorbed in it and the ratio goes towards the 1.23 of full absorption; it is a check of order, not of a value.
- **Lesions:** median 0.991 (0.962-1.040), lower for small lesions (r 0.72 with log mass): electrons escape from lesions of a few mL, as expected.
- **Outliers** (|z| > 4) are small, mostly within 1-2% of 1 in absolute terms, because the spread of each source is tight; the large z come from rest of body (up to 1.25) and GI (up to 1.10), where photon absorption depends on body size and gas, and from one spine at 1.07 (`psma_652cdc5859bf667d_2017-04-28`, metal inside L2-L4, see section 5b). Only one ratio is physically off: the bladder of `psma_8effd73b76ee3191_2016-09-24`, 0.95, a 1.5 mL bladder flagged as outlier in nb09 (almost empty, electrons escape). The two spine cases flagged before are explained by the PET share in L2-L4 (0.26 and 0.065, the latter the lowest of the cohort: L2-L4 at SUV 0.39 against 1.27 in the other vertebrae); after the share they are 0.994 and 1.034. The gland case `psma_4aa52e14d0947256_2017-09-16` is 0.992 with 6.7 g of glands, the smallest of the cohort.

No source is rejected.

## 3. Reciprocity of cross-doses

**Which quantity is symmetric.** For uniform sources the reciprocity theorem makes the *specific* absorbed fraction symmetric, $\Phi(a\leftarrow b)=\Phi(b\leftarrow a)$, so per decay $D(a\leftarrow b)=E\,\Phi(a\leftarrow b)=D(b\leftarrow a)$. The absorbed fraction itself, $D\,m/E$, is not symmetric: $D(a\leftarrow b)\,m_a$ against $D(b\leftarrow a)\,m_b$ differs by $m_a/m_b$. Both are computed below; the first is the check.

Pairs: sources whose target is the same set of voxels (kidneys, liver, spleen, glands, bladder, GI, blood pool). Two limits: sources are PET-weighted, not uniform, so the theorem holds only approximately (activity concentrated near a neighbour breaks it); and there is a single run per source, so no batch SE: far pairs are counted photons from a few hundred thousand decays.

In [ ]:
same = ["kidneys_functional", "liver", "spleen", "salivary_glands", "bladder", "gi_tract", "blood_pool"]
d = source.set_index(["study_id", "source", "target"]).d
self_d = source[source.source == source.target].set_index(["study_id", "source"]).d
rows = []
for s in STUDIES:
    for a, b in itertools.combinations(same, 2):
        if (s, b, a) not in d.index or (s, a, b) not in d.index:
            continue
        d_ab, d_ba = d[(s, b, a)], d[(s, a, b)]
        rows.append(dict(study_id=s, a=a, b=b, d_a_from_b=d_ab, d_b_from_a=d_ba, ratio=d_ab / d_ba if d_ba > 0 else np.nan, ratio_with_mass=d_ab * mass[(s, a)] / (d_ba * mass[(s, b)]) if d_ba > 0 else np.nan, mass_ratio=mass[(s, a)] / mass[(s, b)], smaller_over_self=min(d_ab / self_d[(s, b)], d_ba / self_d[(s, a)])))
recip = pd.DataFrame(rows)
recip.to_csv(OUT / "reciprocity.csv", index=False)
within = lambda v: ((v > 0.8) & (v < 1.25)).mean()
table = recip.groupby(["a", "b"]).agg(n=("ratio", "size"), median=("ratio", "median"), p05=("ratio", lambda v: v.quantile(0.05)), p95=("ratio", lambda v: v.quantile(0.95)), within_0p8_1p25=("ratio", within), median_with_mass=("ratio_with_mass", "median"), median_mass_ratio=("mass_ratio", "median"), cross_over_self=("smaller_over_self", "median"))
display(table.sort_values("cross_over_self", ascending=False).round(3))
print("all pairs: D form within 0.8-1.25", round(within(recip.ratio.dropna()), 3), "| D·m form", round(within(recip.ratio_with_mass.dropna()), 3))
positive = recip[(recip.ratio_with_mass > 0) & np.isfinite(recip.ratio_with_mass)]
print("correlation of log(D·m ratio) with log(mass ratio), pairs with both cross-doses above 0:", round(np.corrcoef(np.log(positive.ratio_with_mass), np.log(positive.mass_ratio))[0, 1], 3))
near = table[table.cross_over_self > 1e-2]
print("pairs whose cross-dose is above 1% of the self-dose:", list(near.index))
zero = recip[(recip.d_a_from_b == 0) | (recip.d_b_from_a == 0)]
print("pairs with a zero cross-dose:", zero.groupby(["a", "b"]).size().to_dict())

**Reading.** Reciprocity is not expected to hold exactly here: the theorem is for uniform sources, while ours are PET-weighted, so activity concentrated on the side facing a neighbour makes $D(a\leftarrow b)$ and $D(b\leftarrow a)$ differ legitimately; and each cross-dose is a single run of 5×10⁵ decays, so far pairs are a count of few photons. With that, the D form holds where it should: liver-spleen 1.005, liver-blood 1.07, spleen-blood 1.07, kidneys-spleen 1.01, kidneys-liver 1.05, with 83-99% of phantoms within 0.8-1.25. The D·m form is off by the mass ratio (correlation 0.98 with log mass ratio), as the theorem says it should be. The pairs that fail are the far ones (glands with anything below the neck; glands-bladder is zero in 179 of 226 phantoms) and kidneys-GI (median 1.34), probably the most asymmetric PET weighting of the list (not verified).

What matters for the doses: **no cross-dose is above 1% of the self-dose of its target** in any phantom, so the pairs that do not reciprocate change organ doses by well under 1%.

## 4. Link to the frozen cohort

**TIA per source.** The cohort's `tia` table holds a TIAC (hours, per unit injected activity) for each of the 9 organ sources of the Monte Carlo and for each lesion (`lesion_j` = label $j+1$ of the phantom). Dose per injected GBq = 1000 × TIAC × dose per unit TIA, Gy/(MBq·h).

**Salivary glands: a link rule.** What it compensates: the PBPK gland TIA is the activity of the glands used in the fit (`use_for_pbpk`: present, not truncated at the top of the PET field of view, with PET), while the Monte Carlo gland source and target are all gland labels 5-8, truncated ones included. The difference is almost only truncated parotids: on the training split 185 parotids are truncated and none is used, all 358 present submandibular glands are used. Putting the PBPK TIA over the larger mask dilutes it by the volume ratio: a first pass gave a Monte Carlo over S-value dose of 0.68 for the glands, and the ratio followed the mass ratio (r 0.99995).

**Rule:** the gland TIA is scaled to the mask at the same concentration, factor = volume of labels 5-8 over the PBPK gland volume, and the excess is taken from the rest of body, which in the cohort is total minus organs, so the total is kept. Why it is correct: (1) the truncated parts are real gland tissue of the same type, which the PBPK did not see only because of the field of view, and the cohort already holds their activity inside the rest of body; (2) self-dose goes with TIA over mass, i.e. with concentration, and the PBPK prediction is a concentration times the PBPK volume, so keeping the concentration keeps the dose that nb19 calibrated on, independently of how much of the gland the FOV cut; (3) measured on the 68Ga PET, a truncated parotid has median 1.06 times the concentration of the complete parotid of the same phantom (29 pairs, quartiles 0.97-1.28), so equal concentration is a fair assumption. The cohort tables are not changed; this is how they enter the Monte Carlo. The effect on the gland dose against its target is below the next cell.

**Bladder of `psma_3959d1c381a5bcd6_2015-11-12`.** The study is cropped at k ≥ 84, which removes the pelvis: no bladder source. Its cohort bladder TIA has no dose map; it is reported, not redistributed.

**Voxel target, as chosen in nb15.** $D(x)=\sum_{c\neq\text{les}} \mathrm{TIA}_c\,B_c(x) + \mathrm{Split}\big(\sum_l \mathrm{TIA}_l w_l\big)(x)$, with $B_c=\mathrm{Split}(w_c)+L_c$: split native kernel (electrons within 10 mm over ρ, photon tail as dose) plus the long-range term $L_c$ of the Monte Carlo by material (σ 10 mm, 40 mm for the rest of body), with metal voxels (HU ≥ 3000) left out of its low-pass (section 5b); lesions per lesion, VSV only. Organ and lesion doses are mass-weighted means of this map.

**How it is computed.** `scripts/monte_carlo/run_cohort_doses.py` (resumable, 3 processes): per phantom, the 9 maps $B_c$ are written to `cohort_doses/basis/<study>/` (float32, zero outside the body) with their means on every target and lesion; per virtual patient and scenario, one transform of the lesion source gives the lesion part on every target and lesion. The organ part is linear in the TIA, so it is formed here from the basis means. The transform shares one forward FFT between the two kernels in single precision (`FastSplit`); against `psma_mc.SplitVsv` the largest difference was 5×10⁻⁷ of the map maximum.

**Disk.** A full map is about 7 MB compressed; 11 828 virtual patients × scenarios would need about 60-80 GB, with 34 GB free. So the maps are not all written: **any map is rebuilt exactly** by `cohort_doses.dose_map(virtual_id, scenario, split)` from the stored basis and one lesion transform (about 3 s). The check below rebuilds maps and compares their means with the tables.

In [ ]:
log_lines = (OUT / "run.log").read_text().splitlines()
print(log_lines[0]); print(log_lines[-1])
parts_done = sorted(p.name[:-21] for p in (OUT / "parts").glob("*_lesion_part.parquet"))
print("phantoms with parts", len(parts_done), "of", len(STUDIES))
basis_size = sum(p.stat().st_size for p in (OUT / "basis").rglob("*.nii.gz")) / 1e9
print(f"basis on disk {basis_size:.1f} GB, {sum(1 for _ in (OUT / 'basis').rglob('B_*.nii.gz'))} maps")

check = pd.read_csv(D.MC_FULL / "independent_check.csv")
s = "psma_bc63defabeea583e_2017-04-22"
stored = D.StoredBasis(s)
assert len(stored.basis) == 9, stored.basis.keys()
tia_test, lesion_tia_test, _ = M.test_tia_v2(s)
rebuilt = stored.dose_map({c: tia_test[c] for c in stored.basis}, {i - 1: lesion_tia_test[i] for i in M.lesion_weights(s)})
rebuilt_means, _ = stored.summarise(rebuilt)
chosen = check[(check.study_id == s) & (check.estimate == "chosen")].set_index("target").gy
print("stored basis (float32 on disk) against the 'chosen' estimate of the independent check, test TIA:", {t: round(rebuilt_means[t] / chosen[t], 5) for t in chosen.index})

In [ ]:
organs_by_split, lesions_by_split, gland_factor = {}, {}, {}
for split in ("train", "test"):
    organs_by_split[split], lesions_by_split[split], gland_factor[split] = D.assemble(split)
    organs_by_split[split].to_parquet(OUT / f"organ_doses_{split}.parquet", index=False)
    lesions_by_split[split].to_parquet(OUT / f"lesion_doses_{split}.parquet", index=False)
    o = organs_by_split[split]
    print(split, "| phantoms", o.study_id.nunique(), "| virtual patient x scenario", o.groupby(["virtual_id", "arm", "scenario"]).ngroups, "| lesion rows", len(lesions_by_split[split]), "| with unplaced lesion TIA", int((o.unplaced_lesion_tiac > 0).sum() / len(D.TARGETS)))
factor = gland_factor["train"]
print("gland link factor, training phantoms: 1 in", int(np.isclose(factor, 1.0).sum()), "of", len(factor), "| changed:", int((~np.isclose(factor, 1.0)).sum()), "| quantiles 50/90/max", factor.quantile([0.5, 0.9]).round(2).to_list(), round(factor.max(), 2), "| among the changed: median", round(factor[~np.isclose(factor, 1.0)].median(), 2))
tia_train = D.load_tia("train")
linked = D.linked_tia(tia_train, factor)
print("rest of body after the gland rule: min TIAC", round(linked.rest_of_body.min(), 4), "h; share moved to the glands, median", round(float(((linked.salivary_glands - tia_train[tia_train.compartment == 'salivary_glands'].set_index(['virtual_id', 'arm', 'scenario']).tiac_h.reindex(linked.index.droplevel(['split', 'study_id'])).to_numpy()) / linked.rest_of_body).median()), 4))
lost = tia_train[(tia_train.study_id == "psma_3959d1c381a5bcd6_2015-11-12") & (tia_train.compartment == "bladder")]
print("psma_3959d1c381a5bcd6_2015-11-12 (no bladder source): bladder TIAC without dose map, median", round(lost.tiac_h.median(), 3), "h")

In [ ]:
glands_meta = []
for s in factor.index:
    for g in json.loads((M.ph.PHANTOMS / s / "meta.json").read_text())["salivary_glands"].values():
        glands_meta.append(dict(study_id=s, gland=g["gland_type"], present=g["present"], truncated=g["truncated"], used=g["use_for_pbpk"]))
glands_meta = pd.DataFrame(glands_meta)
used_count = glands_meta.groupby("study_id").used.sum()
print("training phantoms by number of glands used in the PBPK, with the link factor:")
display(factor.groupby(used_count.reindex(factor.index)).describe()[["count", "min", "50%", "max"]].round(2))
print("glands by type, present, truncated, used:")
display(glands_meta.groupby(["gland", "present", "truncated", "used"]).size().rename("n").to_frame())

target_table = pd.read_csv(D.COHORT / "check_against_targets.csv").set_index("target")
gland_target, gland_tol = target_table.loc["dose_salivary", "published"], target_table.loc["dose_salivary", "tolerance"]
unlinked, _, _ = D.assemble("train", link_glands=False)
rows = []
for label, table in (("with the rule", organs_by_split["train"]), ("without the rule", unlinked)):
    g = table[(table.target == "salivary_glands") & (table.scenario == "A")]
    for arm in ("main", "control"):
        mean = g[g.arm == arm].dose.mean()
        rows.append(dict(link=label, arm=arm, mean_dose_gy_gbq=mean, over_target=mean / gland_target, log_over_target=np.log(mean / gland_target), tolerance=gland_tol))
gland_vs_target = pd.DataFrame(rows).set_index(["link", "arm"])
print(f"gland dose target {gland_target:.3f} Gy/GBq, log tolerance {gland_tol:.3f}; training split, scenario A, mean over virtual patients")
display(gland_vs_target.round(3))
gland_vs_target.to_csv(OUT / "gland_link_vs_target.csv")
del unlinked

**Gland dose with and without the rule.** Same statistic as the nb19 and nb20 checks (mean over virtual patients, scenario A), Monte Carlo map means. The target is the published gland dose that nb19 calibrated the gland agent factor on; the control arm (factor 1) is shown only for scale.

**Reading.** The rule changes 108 of the 180 training phantoms; 71 have all four glands in the PBPK and one has none usable (both volumes are then all glands), factor 1. With two glands used, which is the two submandibular glands with both parotids truncated, the median factor is 2.78 (1.22-5.83); with three, 1.48 (1.24-1.81). The rest of body gives up a median 0.76% of its TIA and never goes below 2.66 h.

Against the target, main arm: **with the rule 1.04 (log +0.04), without it 0.70 (log −0.36).** Both lie inside the log tolerance 0.667, so the rule is not what makes the check pass. It is what makes the Monte Carlo gland dose mean the same as the dose nb19 calibrated the gland agent factor on, a concentration of the PBPK times the self-dose per unit mass. Without it each phantom would be low by its own factor, a bias set by how much of the parotids the PET field of view cut, which a network would have to learn from the field of view. With it, of the remaining +4%, +2.8% was already in the S-value check of nb20 and +1.4% is Monte Carlo over S-value (section 6).

## 5. Organ doses: map means against the raw Monte Carlo

Two organ-level estimates of the same thing on the training split, scenario A. **Map:** the mean of the voxel target (what the network will see). **Raw:** $\sum_c \mathrm{TIA}_c\,D_c(t)$ from the Monte Carlo targets, lesions as the grouped source scaled by the total lesion TIA. They differ by the voxel model of nb15 (independent check: worst 3.4%) and by the grouping of lesions.

In [ ]:
TIA_FRAME = {split: D.linked_tia(D.load_tia(split), gland_factor[split]) for split in ("train",)}
lesion_total = tia_train[tia_train.compartment.str.startswith("lesion_")].groupby(["virtual_id", "arm", "scenario"]).tiac_h.sum()
raw_d = source.pivot_table(index=["study_id", "target"], columns="source", values="d").fillna(0.0)
organs = organs_by_split["train"]
a = organs[organs.scenario == "A"].copy()
tiac = TIA_FRAME["train"].reset_index()
tiac = tiac[tiac.scenario == "A"].set_index(["virtual_id", "arm"])
tiac["lesions"] = lesion_total.xs("A", level="scenario").reindex(tiac.index).fillna(0.0)
rd = raw_d.reindex(pd.MultiIndex.from_frame(a[["study_id", "target"]]))
tv = tiac.reindex(pd.MultiIndex.from_frame(a[["virtual_id", "arm"]]))
a["raw"] = 1e3 * sum(tv[c].to_numpy() * rd[c].to_numpy() for c in D.SOURCES if c in rd.columns)
a["raw_without_lesions"] = 1e3 * sum(tv[c].to_numpy() * rd[c].to_numpy() for c in D.SOURCES if c in rd.columns and c != "lesions")
a["map_over_raw"] = a.dose / a.raw
a["map_over_raw_without_lesions"] = a.organ_part / a.raw_without_lesions
a["lesion_share_map"] = a.lesion_part / a.dose
a["lesion_share_raw"] = (a.raw - a.raw_without_lesions) / a.raw
display(a.groupby("target")[["map_over_raw", "map_over_raw_without_lesions"]].describe(percentiles=[0.05, 0.5, 0.95]).T.round(3))
worst = a.assign(off=np.abs(np.log(a.map_over_raw))).sort_values("off", ascending=False).head(8)
display(worst[["virtual_id", "arm", "target", "map_over_raw", "map_over_raw_without_lesions", "lesion_share_map", "lesion_share_raw"]].round(3))

### 5b. Metal in the long-range term

**What was seen.** After the first full assembly, L2-L4 map over raw without lesions reached 1.8-2.2 on two training phantoms (`psma_0bf6ac34fa0c97c0_2016-12-23`, `psma_9e37770ded7defaa_2015-11-06`, the latter lesion-free), so not a lesion effect. Checked, in this order: untracked bone lesions, high-density material, the mask.

**Cause.** One source, the rest of body: its basis was 4.7 and 3.6 times the raw Monte Carlo on L2-L4, while the kernel part alone was 0.74 and 0.79 of it, as on a normal phantom. The excess was all in $L_c$. Both phantoms have voxels at HU ≥ 3000 touching L2-L4 (CT up to 3400 and 3570 HU, clusters of 30-600 voxels next to the vertebral bodies: spinal hardware). In those voxels the Monte Carlo dose of the rest of body is 30-90 times the kernel dose, so their residual, weighted by the highest density in the phantom, made up 79% and 86% of the residual energy of the bone class; the low-pass (σ 40 mm) then spread it over the neighbouring bone, L2-L4 included. Not the cause: the L2-L4 PET has no hot spot (maximum 0.006 and 0.004 %IA/mL, the same as the other vertebrae), and the masks are contiguous, 243 and 269 mL, within the cohort range (101-393 mL).

**Fix.** Metal voxels (body, HU ≥ 3000, `cohort_doses.METAL_HU`) are left out of the low-pass of $L_c$, through the `near` argument that `long_range_by_material` already had. Their own dose is not a target. Only phantoms with metal change, and nothing else in the target definition: the basis of the 218 phantoms with at least one metal voxel (dental work in most) was rebuilt with `run_cohort_doses.py --metal-basis`, the lesion parts are unchanged (lesions are kernel only). The summaries before the fix are kept in `cohort_doses/parts_before_metal/`.

The cell below shows, on the training phantoms, the basis of the rest-of-body and spine sources on L2-L4 over the raw Monte Carlo, before and after, for the phantoms with metal within 50 mm of L2-L4. On the test split only the count of rebuilt phantoms is given.

In [ ]:
import scipy.ndimage as ndi

metal_rows = []
for s in STUDIES:
    if SPLIT_OF[s] != "train":
        continue
    organs_img = M.load(s, "organs")
    metal = D.metal_mask(s, organs_img)
    if not metal.any():
        continue
    distance = ndi.distance_transform_edt(organs_img != 12, sampling=M.spacing_of(s))
    metal_rows.append(dict(study_id=s, metal_ml=metal.sum() * info.voxel_ml[s], min_distance_mm=float(distance[metal].min()), within_50mm_ml=(metal & (distance <= 50)).sum() * info.voxel_ml[s]))
metal_info = pd.DataFrame(metal_rows).set_index("study_id")
rebuilt = {split: sum((D.OUT / "parts" / f"{s}_metal.done").exists() for s in STUDIES if SPLIT_OF[s] == split) for split in ("train", "test")}
print("phantoms with metal, training:", len(metal_info), "of", int((SPLIT_OF[STUDIES] == "train").sum()), "| rebuilt basis: train", rebuilt["train"], "test", rebuilt["test"])
print("training phantoms with metal within 50 mm of L2-L4:", int((metal_info.min_distance_mm <= 50).sum()))

before = pd.concat([pd.read_parquet(p) for p in sorted((D.OUT / "parts_before_metal").glob("*_basis.parquet"))])
after = pd.concat([pd.read_parquet(D.OUT / "parts" / f"{s}_basis.parquet") for s in STUDIES])
raw_l = source[source.target == "spine_L2_L4"].set_index(["study_id", "source"]).d
pick = lambda f: f[(f.kind == "target") & (f.name == "spine_L2_L4")].set_index(["study_id", "source"]).d
near = metal_info[metal_info.min_distance_mm <= 50].index
compare_metal = pd.DataFrame({"before": pick(before) / raw_l, "after": pick(after) / raw_l}).dropna()
compare_metal = compare_metal[compare_metal.index.get_level_values("study_id").isin(near) & compare_metal.index.get_level_values("source").isin(["rest_of_body", "spine"])]
wide_metal = compare_metal.unstack("source")
wide_metal.columns = [f"{source}_{when}" for when, source in wide_metal.columns]
display(wide_metal.join(metal_info[["min_distance_mm", "within_50mm_ml"]]).round(3))

changed = before.merge(after, on=["study_id", "source", "kind", "name"], suffixes=("_before", "_after"))
changed = changed[changed.study_id.map(SPLIT_OF) == "train"]
changed["ratio"] = changed.d_after / changed.d_before
print("training basis means after over before, all sources and targets (organ targets):")
display(changed[changed.kind == "target"].groupby("name").ratio.describe(percentiles=[0.01, 0.5, 0.99])[["count", "min", "1%", "50%", "99%", "max"]].round(4))

**Reading (sections 5 and 5b).**

- **Map against raw, organ sources only.** Every target is within 0.5% at the median and within ±2.5% at the 5th-95th percentile, except L2-L4 (median 0.972, 5-95% 0.955-0.997) and the grouped lesions. After the fix L2-L4 ranges 0.939-1.078; before it, the two phantoms of section 5b reached 1.8-2.2.
- **The fix, on the 12 training phantoms with metal within 50 mm of L2-L4.** The rest-of-body source on L2-L4 goes from 0.93-4.69 to 0.85-1.15 of the raw Monte Carlo. The spine source of the three phantoms of subject `9e37770ded7defaa`, which have hardware inside L2-L4, goes from 1.07-1.11 to 0.99. What remains (1.13-1.15 on two of them, 0.85 on `psma_995fbaec49f131ce_2016-11-12`) sits on a source that gives 20-55% of the L2-L4 dose, so it is at most about ±7% on the marrow dose of those phantoms: declared, no phantom excluded.
- **Nothing else moves where it should not.** Kidneys, liver, spleen, glands and blood pool are unchanged to 10⁻⁴; bladder and GI within 0.1%. Lesions near metal change only in their crossfire, a median 0.6% of a lesion's dose. The rest-of-body target does change, because metal voxels are part of it. The Monte Carlo deposits 30-90 times the kernel dose per unit mass in them, and the map no longer copies that, so rest-of-body map over raw is now 0.886-1.000 (median 0.994). That is dose to an implant, not to tissue; for tissue the map is the better estimate. The independent check's 0.993 on the rest of body in section 4 has the same cause.
- **The low L2-L4 values with lesions** (down to 0.37) come from the raw estimate, not the map. Raw spreads the total lesion TIA over the grouped lesion source in proportion to the PET, while the map gives each lesion its own TIA as in the cohort. With spine lesions of very different kinetics the grouped estimate is the wrong one (nb15: grouped lesion error median 27%, maximum 130%).

## 6. Monte Carlo doses against the self-dose S-value doses of the nb19 calibration

**What nb19 used.** Self-dose only: $\mathrm{TIAC}\times S_{ref}\,m_{ref}/m$ for kidneys, liver, spleen, glands (EANM 2022, IDAC-Dose 2.1), with the PBPK volumes of the phantom and densities 1.05, 1.05, 1.04, 1.03 g/mL; lesions by local deposition $\Delta_e/m$ at 1.04 g/mL, lesions of at least 0.5 mL, mean over the lesions of a patient, then over patients. Recomputed here from the frozen TIA tables and checked against the cohort's `checks` table.

**Monte Carlo.** Map means of section 5, with the crossfire of every source. Decomposed into: self-dose of the Monte Carlo over the S-value (geometry, mass, PET weighting), and total over self (crossfire). For lesions, the Monte Carlo mass is the CT one (Schneider), so a density-matched ratio is also shown, the Monte Carlo dose times ρ/1.04.

Training split, both arms, scenario A unless said; the calibration statistics are the same as nb19 and nb20 (mean dose over virtual patients).

In [ ]:
checks = pd.read_parquet(D.COHORT / "checks_train.parquet")
volumes = {}
for s in organs.study_id.unique():
    meta = json.loads((M.ph.PHANTOMS / s / "meta.json").read_text())
    c, g = meta["compartments"], meta["groups"]
    used = any(v["use_for_pbpk"] for v in meta["salivary_glands"].values())
    volumes[s] = dict(kidneys=c["kidney_functional"]["volume_ml"], liver=c["liver"]["volume_ml"], spleen=c["spleen"]["volume_ml"], salivary=(g["salivary_glands_pbpk" if used else "salivary_glands_all"]["volume_ml"]) or 1.0)
ORGAN = {"kidneys": "kidneys_functional", "liver": "liver", "spleen": "spleen", "salivary": "salivary_glands"}
tia_wide = tia_train[~tia_train.compartment.str.startswith("lesion_") & tia_train.study_id.isin(organs.study_id.unique())].pivot_table(index=["virtual_id", "arm", "scenario"], columns="compartment", values="tiac_h")
wide = organs.pivot_table(index=["virtual_id", "arm", "scenario"], columns="target", values="dose")
self_part = organs.copy()
self_part["self"] = [getattr(r, f"from_{r.target}") if f"from_{r.target}" in organs.columns else np.nan for r in organs.itertuples()]
self_wide = self_part.pivot_table(index=["virtual_id", "arm", "scenario"], columns="target", values="self")
compare = []
for (vid, arm, scenario), row in tia_wide.iterrows():
    s = vid.rsplit("_", 1)[0]
    out = dict(virtual_id=vid, arm=arm, scenario=scenario, study_id=s)
    for k, t in ORGAN.items():
        out[f"s_{k}"] = C.self_dose(row[t], k, volumes[s][k])
        out[f"mc_{k}"] = wide.loc[(vid, arm, scenario), t]
        out[f"mcself_{k}"] = self_wide.loc[(vid, arm, scenario), t]
    compare.append(out)
compare = pd.DataFrame(compare)
key = checks.set_index(["virtual_id", "arm", "scenario"])
print("S-value doses equal to the cohort checks:", {k: bool(np.allclose(compare[f"s_{k}"], key.loc[list(zip(compare.virtual_id, compare.arm, compare.scenario)), f"dose_{k}"], rtol=1e-6)) for k in ORGAN})


def organ_table(frame):
    rows = {}
    for k in ORGAN:
        ratio = frame[f"mc_{k}"] / frame[f"s_{k}"]
        rows[k] = dict(mean_S=frame[f"s_{k}"].mean(), mean_MC=frame[f"mc_{k}"].mean(), shift_of_means=frame[f"mc_{k}"].mean() / frame[f"s_{k}"].mean() - 1, median_ratio=ratio.median(), p05=ratio.quantile(0.05), p95=ratio.quantile(0.95), self_over_S=(frame[f"mcself_{k}"] / frame[f"s_{k}"]).median(), total_over_self=(frame[f"mc_{k}"] / frame[f"mcself_{k}"]).median(), log_sd_S=np.log(frame[f"s_{k}"]).std(), log_sd_MC=np.log(frame[f"mc_{k}"]).std())
    return pd.DataFrame(rows).T


organ_compare = {arm: organ_table(compare[(compare.arm == arm) & (compare.scenario == "A")]) for arm in ("main", "control")}
for arm, table in organ_compare.items():
    print(arm, "arm, scenario A")
    display(table.round(3))
    table.to_csv(OUT / f"organ_mc_vs_s_{arm}.csv")
by_scenario = compare[compare.arm == "main"].groupby("scenario").apply(lambda f: pd.Series({k: f[f"mc_{k}"].mean() / f[f"s_{k}"].mean() - 1 for k in ORGAN}), include_groups=False)
print("main arm, shift of the mean by scenario")
display(by_scenario.round(3))

In [ ]:
lesions = lesions_by_split["train"].merge(lesion_info[["study_id", "lesion", "volume_ml", "mass_g"]], on=["study_id", "lesion"])
lt = tia_train[tia_train.compartment.str.startswith("lesion_")].copy()
lt["lesion"] = lt.compartment.str[7:].astype(int)
lesions = lesions.merge(lt[["virtual_id", "arm", "scenario", "lesion", "tiac_h"]], on=["virtual_id", "arm", "scenario", "lesion"], how="left")
geometry_ml = {s: np.array(json.loads((M.ph.PHANTOMS / s / "meta.json").read_text())["lesions"]["volume_ml_by_id"], float) for s in lesions.study_id.unique()}
lesions["pbpk_ml"] = [geometry_ml[s][j] for s, j in zip(lesions.study_id, lesions.lesion)]
print("lesion volume of the phantom against the one in the PBPK geometry: max relative difference", float(np.abs(lesions.volume_ml / lesions.pbpk_ml - 1).max()))
lesions["s"] = C.self_dose(lesions.tiac_h, "lesions", lesions.pbpk_ml)
lesions["rho"] = lesions.mass_g / lesions.volume_ml
lesions["ratio"] = lesions.dose / lesions.s
lesions["ratio_self"] = lesions.lesion_part / lesions.s
lesions["ratio_density_matched"] = lesions.ratio * lesions.rho / C.DENSITY["lesions"]
lesions["crossfire_share"] = lesions.organ_part / lesions.dose
kept = lesions[(lesions.pbpk_ml >= C.MIN_LESION_ML) & (lesions.scenario == "A")]
for arm in ("main", "control"):
    k = kept[kept.arm == arm]
    per_patient = k.groupby("virtual_id").agg(s=("s", "mean"), mc=("dose", "mean"), mc_matched=("dose", lambda v: float((v * k.loc[v.index, "rho"] / C.DENSITY["lesions"]).mean())))
    print(f"{arm}: {len(k)} lesions >= 0.5 mL, {len(per_patient)} virtual patients | calibration statistic (mean of patient means): S {per_patient.s.mean():.3f}, MC {per_patient.mc.mean():.3f} Gy/GBq, shift {per_patient.mc.mean() / per_patient.s.mean() - 1:+.3f}; density-matched {per_patient.mc_matched.mean() / per_patient.s.mean() - 1:+.3f}")
    print(f"   per lesion: median ratio {k.ratio.median():.3f} (5-95% {k.ratio.quantile(0.05):.3f}-{k.ratio.quantile(0.95):.3f}), self part {k.ratio_self.median():.3f}, density-matched {k.ratio_density_matched.median():.3f} ({k.ratio_density_matched.quantile(0.05):.3f}-{k.ratio_density_matched.quantile(0.95):.3f}); crossfire share median {k.crossfire_share.median():.3f}, p95 {k.crossfire_share.quantile(0.95):.3f}")
main_kept = kept[kept.arm == "main"]
display(main_kept.groupby(pd.cut(main_kept.rho, [0, 1.0, 1.06, 1.15, 1.3, 3])).agg(lesions=("ratio", "size"), median_ratio=("ratio", "median"), density_matched=("ratio_density_matched", "median"), median_volume_ml=("volume_ml", "median")).round(3))
display(main_kept.groupby(pd.cut(main_kept.volume_ml, [0.5, 1, 3, 10, 100, 1e4])).agg(lesions=("ratio", "size"), median_ratio=("ratio", "median"), density_matched=("ratio_density_matched", "median"), crossfire_share=("crossfire_share", "median")).round(3))
small = lesions[(lesions.pbpk_ml < C.MIN_LESION_ML) & (lesions.scenario == "A") & (lesions.arm == "main")]
print(f"lesions below 0.5 mL (not in the calibration): {len(small)}, median density-matched ratio {small.ratio_density_matched.median():.3f}")
lesion_summary = main_kept.groupby(pd.cut(main_kept.rho, [0, 1.0, 1.06, 1.15, 1.3, 3]), observed=True).ratio.median()
main_kept[["virtual_id", "study_id", "lesion", "volume_ml", "rho", "s", "dose", "ratio", "ratio_density_matched", "crossfire_share"]].to_parquet(OUT / "lesion_mc_vs_s_main_A.parquet", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
ax = axes[0]
for i, (k, label) in enumerate(ORGAN.items()):
    f = compare[(compare.arm == "main") & (compare.scenario == "A")]
    r = f[f"mc_{k}"] / f[f"s_{k}"]
    parts = ax.violinplot(r, positions=[i], widths=0.7, showextrema=False)
    for p in parts["bodies"]:
        p.set_facecolor(COLORS[i]); p.set_alpha(0.55); p.set_edgecolor("none")
    ax.scatter([i], [f[f"mc_{k}"].mean() / f[f"s_{k}"].mean()], color=INK, s=14, zorder=3)
ax.axhspan(0.9, 1.1, color="#e6e6e3", zorder=0)
ax.axhline(1, color=MUTED, lw=0.8)
ax.set_xticks(range(len(ORGAN)), list(ORGAN))
ax.set_ylabel("Monte Carlo / S-value self-dose")
ax.set_title("Organs, main arm, scenario A (dot: ratio of means)", fontsize=9)
ax = axes[1]
ax.scatter(main_kept.rho, main_kept.ratio, s=3, alpha=0.25, color=COLORS[0], linewidths=0, label="Monte Carlo / S")
ax.scatter(main_kept.rho, main_kept.ratio_density_matched, s=3, alpha=0.25, color=COLORS[1], linewidths=0, label="× ρ/1.04")
ax.axhspan(0.9, 1.1, color="#e6e6e3", zorder=0)
ax.axhline(1, color=MUTED, lw=0.8)
ax.set_xlabel("lesion density in the Monte Carlo geometry (g/mL)")
ax.set_title("Lesions ≥ 0.5 mL, main arm, scenario A", fontsize=9)
ax.legend(frameon=False, markerscale=4, fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "mc_vs_s_value.png", dpi=150)
plt.show()

**Reading.**

- **Organs: the shifts are crossfire.** Monte Carlo over S-value, shift of the means, main arm, scenario A: kidneys +1.9%, liver +3.9%, spleen +7.4%, glands +1.4%. The Monte Carlo self-dose alone is 1.006-1.015 of the S-value, so the shift is the crossfire that the self-dose-only calibration of nb19 ignored (total over self 1.008, 1.027, 1.062, 1.002). The spleen and liver are small or low-uptake targets next to the kidneys, the rest of body and each other, so they receive the most. The shifts are the same in every scenario (within 0.002), and the log SD of dose does not change (0.386-0.502 against 0.389-0.503).
- **Lesions: the −5.2% is density.** Calibration statistic −5.2% (main) and −5.5% (control); per lesion, median 0.88. The S-value uses 1.04 g/mL, while the Monte Carlo has the CT density, and half of the lesions are above 1.15 g/mL (bone): the ratio is 0.77 above 1.3 g/mL and 1.08 below 1.0 g/mL. **At equal density the difference is −0.5%**, 0.996 per lesion (0.975-1.030), with a small trend in volume: 0.98 below 1 mL, where electrons escape, and 1.03 above 100 mL, where photons are absorbed. Crossfire is a median 0.6% of a lesion's dose. Which density the published lesion doses imply is not known; the 5% between the two is declared.
- **Within the nb19 tolerances.** Monte Carlo over target, main arm: kidneys 1.17, liver 1.04, spleen 1.08, glands 1.04, lesions 1.21. In log: 0.16, 0.03, 0.08, 0.04, 0.19, against tolerances 0.38, 0.25, 0.36, 0.67, 0.42. **No recalibration; the cohort stays frozen.** The network's dose targets are the Monte Carlo doses of this notebook; the S-value doses stay what the calibration was done on.

## 7. Voxel dose maps

A map of the training cohort, rebuilt from the stored basis, and the check that its means equal the tables of section 4. One coronal slice through the kidneys, Gy/GBq, logarithmic scale.

In [ ]:
example = "psma_bc63defabeea583e_2017-04-22_d00"
dose = D.dose_map(example, "A", "train", "main")
stored = D.StoredBasis(example.rsplit("_", 1)[0])
means, lesion_means = stored.summarise(dose)
table = organs_by_split["train"]
row = table[(table.virtual_id == example) & (table.arm == "main") & (table.scenario == "A")].set_index("target").dose
print("map means over the table:", {t: round(means[t] / row[t], 6) for t in row.index if row[t] > 0})
lt_rows = lesions_by_split["train"]
lt_rows = lt_rows[(lt_rows.virtual_id == example) & (lt_rows.arm == "main") & (lt_rows.scenario == "A")].set_index("lesion").dose
print("lesions: max |map/table - 1|", float(np.max(np.abs(lesion_means[lt_rows.index] / lt_rows.to_numpy() - 1))))
organs_img = M.load(example.rsplit("_", 1)[0], "organs")
y = int(np.argmax((organs_img == 1).sum(axis=(0, 2))))
sl = np.where(organs_img[:, y, :] > 0, dose[:, y, :], np.nan).T[::-1]
spacing = M.spacing_of(example.rsplit("_", 1)[0])
fig, ax = plt.subplots(figsize=(4, 6))
im = ax.imshow(np.log10(np.maximum(sl, 1e-3)), cmap="magma", vmin=-2, vmax=1.5, aspect=spacing[2] / spacing[0])
ax.set_axis_off()
cb = fig.colorbar(im, ax=ax, shrink=0.7)
cb.set_label("log10 dose (Gy/GBq)")
ax.set_title(example, fontsize=7)
fig.tight_layout()
fig.savefig(OUT / "dose_map_example.png", dpi=150)
plt.show()

## Report

**Run.** The full Monte Carlo of nb15's design is complete: 227 phantoms, 2245 source runs of 5×10⁵ decays plus 25 expected skips, no failure left. Self-doses agree with mass-scaled S-values within 1-2% for the organs with an S-value. Against the local-deposition references they are within 3%, except the rest of body (1.09, photons absorbed in a large mass) and one 1.5 mL bladder (0.95). Reciprocity holds where it should, given PET-weighted sources and single runs; no cross-dose is above 1% of the self-dose of its target.

**Link to the cohort.** Tables are unchanged (manifest 14 of 14). Gland TIA is scaled to the Monte Carlo gland mask at the PBPK concentration (108 of 180 training phantoms, almost all from truncated parotids), with the excess taken from the rest of body. Gland dose over target: 1.04 with the rule, 0.70 without. One cropped study has a bladder TIA and no bladder map (median 1.6 h, reported).

**Voxel target.** As chosen in nb15, with one change: metal voxels (HU ≥ 3000) are left out of the low-pass of the long-range term. Spinal hardware next to L2-L4 had inflated the rest-of-body dose there by up to 4.7 times on training phantoms (two cases seen in the first assembly). The organ basis of the 218 phantoms with metal was rebuilt; organ targets other than L2-L4 and the rest of body are unchanged. Remaining marrow error on the hardware phantoms is at most about ±7%, declared. Map means agree with the raw Monte Carlo within 0.5% at the median on every organ target. Any voxel map is rebuilt exactly from the stored basis (12.9 GB) and one lesion transform.

**Against the calibration.** Organ doses are +1.4% to +7.4% above the S-value doses of nb19, the crossfire it ignored. Lesions are −5.2%, a density effect (−0.5% at equal density). All doses are within the nb19 tolerances (largest: lesions, log 0.19 against 0.42). No recalibration; the cohort stays frozen.

**Outputs** in `data/processed/psma/cohort_doses/`: `organ_doses_{train,test}.parquet`, `lesion_doses_{train,test}.parquet` (Gy/GBq per virtual patient, arm, scenario, with the part from every source); `basis/` and `parts/`; `parts_before_metal/` (basis summaries before the metal fix); `gland_link_vs_target.csv`, `self_dose_vs_s_values.csv`, `reciprocity.csv`, `organ_mc_vs_s_{main,control}.csv`, `lesion_mc_vs_s_main_A.parquet`, figures. Test doses are computed and saved; no statistic of them is shown.

**Limitations.** Marrow dose on the phantoms with spinal hardware (±7%); rest-of-body dose leaves out the dose to implants; which lesion density the published doses imply (5%); kidney-GI reciprocity 1.34, not explained; per-voxel Monte Carlo noise of the long-range term as in nb15.